# NeuralBudget Quick Start - Python Client (5 Minutes Each)

This notebook demonstrates how to evaluate SLOs using the NeuralBudget Python client library.

**What you'll learn:**
- Evaluate HTTP SLOs for availability & latency
- Monitor ML model drift & confidence
- Track GenAI TTFT & throughput
- Use Python for programmatic SLO evaluation

Copy-paste examples for each use case - no setup required!

## Section 1️⃣: HTTP Availability & Latency SLO

**Use Case:** Monitor REST API uptime and response time

**What this does:**
- Check if 99.9% of requests succeed
- Ensure P99 latency < 200ms
- Alert on error budget burn rate

**Time:** ~2 minutes

In [ ]:
# Copy-paste: Install NeuralBudget (run once)
import subprocess
import sys

# Install if not already installed
try:
    from neuralbudget import NeuralBudgetClient
    print("✓ NeuralBudget already installed")
except ImportError:
    print("Installing neuralbudget...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "neuralbudget"])
    from neuralbudget import NeuralBudgetClient
    print("✓ Installation complete")

In [ ]:
# Copy-paste: HTTP SLO Configuration (YAML as dict)
http_slo_config = {
    "schema_version": 1,
    "mode": "http",
    "profile": "balanced",
    "params": {
        "latency_threshold_ms": 200.0,
        "availability_threshold": 0.999,
        "latency_percentile": 0.99
    }
}

print("✓ HTTP SLO Configuration loaded")
print(f"  Target: {http_slo_config['params']['availability_threshold']*100:.1f}% availability")
print(f"  Latency: P{int(http_slo_config['params']['latency_percentile']*100)} < {int(http_slo_config['params']['latency_threshold_ms'])}ms")

In [ ]:
# Copy-paste: HTTP Sample Metrics (5-minute window)
http_sample = {
    "timestamp": 1704067200,
    "service": "payment-api",
    "measurement_window": "5m",
    "requests": {
        "total": 50000,
        "successful": 49950,
        "failed": 50
    },
    "latency": {
        "p50_ms": 45.2,
        "p90_ms": 125.8,
        "p99_ms": 187.5,      # ✓ Below 200ms threshold
        "p99_9_ms": 250.3,
        "max_ms": 892.1
    },
    "errors": {
        "connection_refused": 5,
        "timeout": 0,
        "service_unavailable": 35,
        "auth_failed": 10
    }
}

# Calculate availability
availability = http_sample["requests"]["successful"] / http_sample["requests"]["total"]
print(f"✓ Sample HTTP Metrics (5m window)")
print(f"  Availability: {availability*100:.2f}%")
print(f"  P99 Latency: {http_sample['latency']['p99_ms']}ms")
print(f"  Success Rate: {http_sample['requests']['successful']}/{http_sample['requests']['total']}")

In [ ]:
# Copy-paste: Evaluate HTTP SLO
from neuralbudget import NeuralBudgetClient

# Initialize client with HTTP config
client = NeuralBudgetClient()
client.load_config(http_slo_config)

# Run evaluation
result = client.evaluate(http_sample)

# Display results
print("\n" + "="*50)
if result['passed']:
    print("✓ SLO PASS")
else:
    print("✗ SLO FAIL")
print("="*50)

print(f"\nAvailability:")
print(f"  Current: {result.get('availability', 'N/A')*100:.2f}%")
print(f"  Target:  99.90%")
print(f"  Status:  {'✓ PASS' if result.get('availability', 0) >= 0.999 else '✗ FAIL'}")

print(f"\nLatency (P99):")
print(f"  Current: {http_sample['latency']['p99_ms']}ms")
print(f"  Target:  <200ms")
print(f"  Status:  {'✓ PASS' if http_sample['latency']['p99_ms'] < 200 else '✗ FAIL'}")

print(f"\nError Budget:")
print(f"  Remaining: {(1 - (50/50000)) * 100:.2f}%")
print(f"  Window: 30 days")
print(f"  Status: ✓ All checks passed")

## Section 2️⃣: ML Model Drift & Confidence Monitoring

**Use Case:** Monitor ML model accuracy, feature drift, and prediction confidence

**What this does:**
- Check model accuracy >= 92%
- Monitor feature drift < 15%
- Ensure prediction confidence >= 80%
- Alert on model degradation

**Time:** ~2 minutes

In [ ]:
# Copy-paste: ML SLO Configuration
ml_slo_config = {
    "schema_version": 1,
    "mode": "ml",
    "profile": "strict_accuracy",
    "params": {
        "accuracy_threshold": 0.92,
        "latency_threshold_ms": 500.0,
        "drift_threshold": 0.05,
        "min_confidence": 0.80
    }
}

print("✓ ML SLO Configuration loaded")
print(f"  Accuracy: >= {ml_slo_config['params']['accuracy_threshold']*100:.0f}%")
print(f"  Drift: <= {ml_slo_config['params']['drift_threshold']*100:.1f}%")
print(f"  Confidence: >= {ml_slo_config['params']['min_confidence']*100:.0f}%")
print(f"  Latency: < {int(ml_slo_config['params']['latency_threshold_ms'])}ms")

In [ ]:
# Copy-paste: ML Sample Metrics with Drift & Confidence
ml_sample = {
    "timestamp": 1704067200,
    "service": "recommendation-model",
    "measurement_window": "5m",
    "model_metrics": {
        "accuracy": 0.945,          # ✓ Above 92% threshold
        "precision": 0.952,
        "recall": 0.938,
        "f1_score": 0.945,
        "auc_roc": 0.978,
        "log_loss": 0.125
    },
    "confidence": {
        "mean_confidence": 0.88,    # ✓ Above 80% threshold
        "min_confidence": 0.72,
        "predictions_below_threshold": 45,
        "total_predictions": 50000
    },
    "drift": {
        "feature_drift_score": 0.032,  # ✓ Below 5% threshold (actually 3.2%)
        "target_drift_score": 0.045,
        "concept_drift_detected": False
    },
    "latency": {
        "mean_ms": 85.3,
        "p99_ms": 142.5,
        "max_ms": 298.1
    },
    "inference": {
        "total": 50000,
        "successful": 49950,
        "errors": 50
    }
}

print("✓ Sample ML Metrics (5m window)")
print(f"  Accuracy: {ml_sample['model_metrics']['accuracy']*100:.1f}%")
print(f"  Drift: {ml_sample['drift']['feature_drift_score']*100:.1f}%")
print(f"  Confidence: {ml_sample['confidence']['mean_confidence']*100:.0f}%")
print(f"  Predictions: {ml_sample['inference']['successful']}/{ml_sample['inference']['total']}")

In [ ]:
# Copy-paste: Evaluate ML SLO
client_ml = NeuralBudgetClient()
client_ml.load_config(ml_slo_config)
result_ml = client_ml.evaluate(ml_sample)

print("\n" + "="*50)
if result_ml['passed']:
    print("✓ SLO PASS - Model performing well")
else:
    print("✗ SLO FAIL - Model degradation detected")
print("="*50)

print(f"\nAccuracy:")
print(f"  Current: {ml_sample['model_metrics']['accuracy']*100:.1f}%")
print(f"  Target:  ≥92%")
print(f"  Status:  {'✓ PASS' if ml_sample['model_metrics']['accuracy'] >= 0.92 else '✗ FAIL'}")

print(f"\nFeature Drift:")
print(f"  Current: {ml_sample['drift']['feature_drift_score']*100:.2f}%")
print(f"  Target:  <5%")
print(f"  Status:  {'✓ PASS' if ml_sample['drift']['feature_drift_score'] < 0.05 else '✗ FAIL'}")

print(f"\nPrediction Confidence:")
print(f"  Current: {ml_sample['confidence']['mean_confidence']*100:.0f}%")
print(f"  Target:  ≥80%")
print(f"  Status:  {'✓ PASS' if ml_sample['confidence']['mean_confidence'] >= 0.80 else '✗ FAIL'}")

print(f"\nModel Info:")
print(f"  Precision: {ml_sample['model_metrics']['precision']*100:.1f}%")
print(f"  Recall: {ml_sample['model_metrics']['recall']*100:.1f}%")
print(f"  Latency P99: {ml_sample['latency']['p99_ms']}ms")

## Section 3️⃣: GenAI Token-Per-Second & Time-To-First-Token

**Use Case:** Monitor LLM endpoints, RAG systems, and AI workloads

**What this does:**
- Track TTFT (Time To First Token) < 1 second
- Monitor throughput >= 50 tokens/sec
- Check availability 99.9%
- Track semantic quality >= 85%

**Time:** ~2 minutes

In [ ]:
# Copy-paste: GenAI SLO Configuration
genai_slo_config = {
    "schema_version": 1,
    "mode": "genai",
    "profile": "strict_latency",
    "params": {
        "ttft_threshold_ms": 1000.0,
        "throughput_threshold_tokens_per_sec": 50.0,
        "availability_threshold": 0.999,
        "semantic_quality_threshold": 0.85
    }
}

print("✓ GenAI SLO Configuration loaded")
print(f"  TTFT: < {int(genai_slo_config['params']['ttft_threshold_ms'])}ms")
print(f"  Throughput: >= {int(genai_slo_config['params']['throughput_threshold_tokens_per_sec'])} tokens/sec")
print(f"  Availability: >= {genai_slo_config['params']['availability_threshold']*100:.1f}%")
print(f"  Quality: >= {genai_slo_config['params']['semantic_quality_threshold']*100:.0f}%")

In [ ]:
# Copy-paste: GenAI Sample Metrics with TTFT & Throughput
genai_sample = {
    "timestamp": 1704067200,
    "service": "gpt-api-endpoint",
    "measurement_window": "5m",
    "requests": {
        "total": 1000,
        "successful": 999,
        "failed": 1,
        "rate_limited": 0
    },
    "latency": {
        "ttft_ms": 850.5,           # ✓ Below 1000ms threshold
        "ttft_p99_ms": 950.2,
        "total_time_to_completion_ms": 2450.3,
        "mean_ttft_ms": 780.1
    },
    "throughput": {
        "tokens_per_sec": 65.3,     # ✓ Above 50 tokens/sec threshold
        "mean_tokens_per_sec": 62.1,
        "min_tokens_per_sec": 45.8,
        "p99_tokens_per_sec": 70.2
    },
    "tokens": {
        "total_input_tokens": 450000,
        "total_output_tokens": 250000,
        "total_tokens": 700000,
        "estimated_cost_usd": 7.00
    },
    "quality": {
        "semantic_quality_score": 0.92,  # ✓ Above 85% threshold
        "hallucination_rate": 0.02,
        "relevance_score": 0.91
    },
    "model_info": {
        "model_name": "gpt-4-turbo",
        "model_version": "2024-01"
    }
}

print("✓ Sample GenAI Metrics (5m window)")
print(f"  TTFT: {genai_sample['latency']['ttft_ms']}ms")
print(f"  Throughput: {genai_sample['throughput']['tokens_per_sec']} tokens/sec")
print(f"  Availability: {genai_sample['requests']['successful']/genai_sample['requests']['total']*100:.1f}%")
print(f"  Quality Score: {genai_sample['quality']['semantic_quality_score']*100:.0f}%")
print(f"  Hallucination Rate: {genai_sample['quality']['hallucination_rate']*100:.1f}%")

In [ ]:
# Copy-paste: Evaluate GenAI SLO
client_genai = NeuralBudgetClient()
client_genai.load_config(genai_slo_config)
result_genai = client_genai.evaluate(genai_sample)

print("\n" + "="*50)
if result_genai['passed']:
    print("✓ SLO PASS - LLM endpoint performing well")
else:
    print("✗ SLO FAIL - LLM performance degradation")
print("="*50)

print(f"\nTime-To-First-Token (TTFT):")
print(f"  Current: {genai_sample['latency']['ttft_ms']}ms")
print(f"  Target:  <1000ms")
print(f"  Status:  {'✓ PASS' if genai_sample['latency']['ttft_ms'] < 1000 else '✗ FAIL'}")

print(f"\nThroughput:")
print(f"  Current: {genai_sample['throughput']['tokens_per_sec']:.1f} tokens/sec")
print(f"  Target:  ≥50 tokens/sec")
print(f"  Status:  {'✓ PASS' if genai_sample['throughput']['tokens_per_sec'] >= 50 else '✗ FAIL'}")

print(f"\nAvailability:")
availability = genai_sample["requests"]["successful"] / genai_sample["requests"]["total"]
print(f"  Current: {availability*100:.2f}%")
print(f"  Target:  99.9%")
print(f"  Status:  {'✓ PASS' if availability >= 0.999 else '✗ FAIL'}")

print(f"\nQuality Score:")
print(f"  Current: {genai_sample['quality']['semantic_quality_score']*100:.0f}%")
print(f"  Target:  ≥85%")
print(f"  Status:  {'✓ PASS' if genai_sample['quality']['semantic_quality_score'] >= 0.85 else '✗ FAIL'}")

print(f"\nCost Tracking:")
print(f"  Tokens Used: {genai_sample['tokens']['total_tokens']:,}")
print(f"  Estimated Cost: ${genai_sample['tokens']['estimated_cost_usd']:.2f}")

## Section 4️⃣: Prometheus Integration & Rule Generation

**Use Case:** Generate Prometheus alerting rules from SLO definitions

**What this does:**
- Convert SLO config to Prometheus alerting rules
- Generate burn rate alerts for multi-window monitoring
- Deploy rules to Kubernetes PrometheusRules
- Monitor alerts firing in real-time

**Time:** ~3 minutes

In [ ]:
# Generate Prometheus Rules from HTTP SLO
# Command: neuralbudget gen-rules slo.yaml > prometheus-rules.yaml

# Example of generated rules (what you'd get):
prometheus_rules_example = """
groups:
- name: neuralbudget_slo_rules
  interval: 30s
  rules:
  - alert: SLOAvailabilityBurnFast
    expr: |
      (1 - (rate(requests_success_total[1h]) / 
             rate(requests_total[1h]))) > 0.001
    for: 5m
    labels:
      severity: critical
      slo: quickstart-api
    annotations:
      summary: "API availability burning error budget too fast"
      
  - alert: SLOLatencyBurnFast
    expr: |
      (histogram_quantile(0.99, rate(request_duration_bucket[1h])) / 0.200) > 1.0
    for: 5m
    labels:
      severity: warning
      slo: quickstart-api
    annotations:
      summary: "API latency P99 exceeds SLO threshold"
"""

print("✓ Example Prometheus Rules Generated")
print(prometheus_rules_example)

### Step-by-Step: Deploy Prometheus Rules to Kubernetes

**Step 1: Generate Rules**
```bash
# Generate rules from SLO definition
neuralbudget gen-rules examples/quickstart/http-slo/slo.yaml > prometheus-rules.yaml
```

**Step 2: Review Generated Rules**
```bash
# View the generated rules
cat prometheus-rules.yaml
```

**Step 3: Deploy to Kubernetes**
```bash
# Apply as PrometheusRule CRD
kubectl apply -f prometheus-rules.yaml -n monitoring

# Or deploy to Prometheus config directory
cp prometheus-rules.yaml /etc/prometheus/rules/
curl -X POST http://localhost:9090/-/reload
```

**Step 4: View Alerts in Prometheus UI**
- Open: `http://prometheus:9090/alerts`
- Look for alerts like `SLOAvailabilityBurnFast`, `SLOLatencyBurnFast`
- Green = SLO passing ✓
- Red = SLO failing ✗

## Section 5️⃣: Python Client Library Usage

**Use Case:** Programmatic SLO evaluation in your application

**What this does:**
- Import NeuralBudgetClient
- Load SLO configuration
- Evaluate metrics samples
- Parse pass/fail results
- Integrate into monitoring pipelines

**Time:** ~2 minutes

In [ ]:
# Pattern 1: Simple Pass/Fail Check
print("="*60)
print("Pattern 1: Simple Pass/Fail Check")
print("="*60)

client = NeuralBudgetClient()
client.load_config(http_slo_config)
result = client.evaluate(http_sample)

if result['passed']:
    print("✓ SLO PASSED")
else:
    print("✗ SLO FAILED")

# Pattern 2: Extract Specific Metrics
print("\n" + "="*60)
print("Pattern 2: Extract Specific Metrics")
print("="*60)

metrics = {
    'passed': result['passed'],
    'availability': result.get('availability', 0),
    'latency_p99_ms': http_sample['latency']['p99_ms'],
    'error_budget_remaining': (http_sample['requests']['successful'] / http_sample['requests']['total']) * 100
}

for key, value in metrics.items():
    print(f"  {key}: {value}")

In [ ]:
# Pattern 3: Batch Evaluation Over Time Series
print("\n" + "="*60)
print("Pattern 3: Batch Evaluation (Multiple Samples)")
print("="*60)

# Simulate multiple 5-minute windows
import random

samples = [
    {**http_sample, "requests": {"total": 50000, "successful": random.randint(49800, 50000), "failed": random.randint(0, 200)}},
    {**http_sample, "requests": {"total": 50000, "successful": random.randint(49800, 50000), "failed": random.randint(0, 200)}},
    {**http_sample, "requests": {"total": 50000, "successful": random.randint(49800, 50000), "failed": random.randint(0, 200)}},
]

results_batch = []
for i, sample in enumerate(samples):
    result = client.evaluate(sample)
    passed = result['passed']
    success_rate = sample['requests']['successful'] / sample['requests']['total']
    results_batch.append({'window': i+1, 'passed': passed, 'success_rate': success_rate})
    print(f"  Window {i+1}: {'✓ PASS' if passed else '✗ FAIL'} ({success_rate*100:.2f}% success)")

print(f"\nSummary: {sum(1 for r in results_batch if r['passed'])}/{len(results_batch)} windows passed")

In [ ]:
# Pattern 4: Alerting Integration
print("\n" + "="*60)
print("Pattern 4: Alerting Integration")
print("="*60)

def send_alert(service, slo_name, status):
    """Mock function - replace with your alerting system"""
    print(f"  📤 Alert: {service} {slo_name} {status}")
    # In production: send to Slack, PagerDuty, etc.

# Evaluate and trigger alerts
result = client.evaluate(http_sample)

if result['passed']:
    print(f"✓ HTTP API SLO is PASSING")
else:
    print(f"✗ HTTP API SLO is FAILING")
    send_alert(service="payment-api", slo_name="HTTP-Availability", status="CRITICAL")

# Pattern 5: Config from YAML File
print("\n" + "="*60)
print("Pattern 5: Load Config from File")
print("="*60)

import yaml

# Example YAML config
yaml_config = """
schema_version: 1
mode: http
profile: balanced
params:
  latency_threshold_ms: 200.0
  availability_threshold: 0.999
"""

config_dict = yaml.safe_load(yaml_config)
print(f"✓ Loaded config: {config_dict['mode']} mode")
print(f"  Availability target: {config_dict['params']['availability_threshold']*100:.1f}%")

# You can also load from file:
# with open('slo.yaml') as f:
#     config = yaml.safe_load(f)
#     client.load_config(config)

## Summary & Next Steps

### What You Learned ✅

| Section | Use Case | Key Metric |
|---------|----------|-----------|
| 1️⃣ HTTP | API availability & latency | P99 latency, error rate |
| 2️⃣ ML | Model performance & drift | Accuracy, drift score |
| 3️⃣ GenAI | LLM endpoints | TTFT, throughput |
| 4️⃣ Prometheus | Alert generation | Burn rate rules |
| 5️⃣ Python | Programmatic usage | Pass/fail decisions |

### Next Steps 🚀

1. **Copy-paste your use case** from the examples above
2. **Customize thresholds** to match your SLOs
3. **Load real metrics** from your monitoring system
4. **Deploy to production** using Kubernetes + Prometheus
5. **Set up alerting** to Slack/PagerDuty

### Common Next Steps

- 📖 [Full SLO Guide](https://github.com/pristley/NeuralBudget/tree/main/docs/guides)
- 🔧 [Production Deployment](https://github.com/pristley/NeuralBudget/tree/main/docs/guides/production-deployment.md)
- 🤝 [Kubernetes Integration](https://github.com/pristley/NeuralBudget/tree/main/docs/guides/kubernetes-integration.md)
- 📊 [API Reference](https://github.com/pristley/NeuralBudget/tree/main/docs/reference/api.md)
- 💬 [Get Help](https://github.com/pristley/NeuralBudget/discussions)

### Troubleshooting

**Metrics not evaluating?**
- Check YAML syntax in SLO config
- Verify sample data has all required fields
- Review [API Reference](https://github.com/pristley/NeuralBudget/tree/main/docs/reference/api.md)

**Not passing when expected?**
- Lower thresholds in SLO config
- Check calculation of availability/latency
- Inspect detailed logs: `DEBUG=1 neuralbudget eval ...`

### Get Involved 🎉

- ⭐ [Star on GitHub](https://github.com/pristley/NeuralBudget)
- 🐛 [Report Issues](https://github.com/pristley/NeuralBudget/issues)
- 💡 [Feature Requests](https://github.com/pristley/NeuralBudget/issues/new)
- 🤝 [Contribute](https://github.com/pristley/NeuralBudget/blob/main/CONTRIBUTING.md)